In [3]:
from pathlib import Path
from datetime import datetime, timezone
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Files generated by the revised modelling notebook (not the original 80/20 run)
PREDICTION_DIR = Path("models_validation_fixed")
OUTPUT_DIR = Path("database_upload_review")
OUTPUT_DIR.mkdir(exist_ok=True)

load_dotenv(override=True)
TASKS = {
    "NO2_next_hour": {"pollutant": "NO2", "horizon_hours": 1},
    "PM10_next_hour": {"pollutant": "PM10", "horizon_hours": 1},
    "PM10_next_3h_average": {"pollutant": "PM10", "horizon_hours": 3},
    "PM10_next_24h_average": {"pollutant": "PM10", "horizon_hours": 24},
}
print("Prediction folder exists:", PREDICTION_DIR.is_dir())
print("Output folder:", OUTPUT_DIR.resolve())

Prediction folder exists: True
Output folder: C:\Users\suraj\Downloads\database_upload_review


                Load revised model results and check their integrity



In [5]:
summary_file = PREDICTION_DIR / "revised_retrospective_metrics.csv"
if not summary_file.is_file():
    raise FileNotFoundError(f"Run the revised modelling notebook first: {summary_file}")
summary = pd.read_csv(summary_file)
required_summary = {"Task", "Selected on validation", "MAE", "RMSE", "R2"}
if not required_summary.issubset(summary.columns):
    raise ValueError(f"Missing metrics columns: {required_summary - set(summary.columns)}")
if set(summary["Task"]) != set(TASKS):
    raise ValueError("Results must have exactly the four expected forecasting tasks")
if summary["Task"].duplicated().any():
    raise ValueError("Duplicate tasks in final summary")
display(summary)

predictions_by_task = {}
for task in TASKS:
    path = PREDICTION_DIR / f"{task}_retrospective_test_predictions.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Missing revised predictions: {path}")
    d = pd.read_csv(path)
    if not {"datetime", "actual", "predicted"}.issubset(d.columns):
        raise ValueError(f"{task}: expected datetime, actual, predicted")
    d = d[["datetime", "actual", "predicted"]].copy()
    d["datetime"] = pd.to_datetime(d["datetime"], errors="raise")
    for col in ("actual", "predicted"):
        d[col] = pd.to_numeric(d[col], errors="raise")
    if d.empty or d["datetime"].isna().any() or d["datetime"].duplicated().any():
        raise ValueError(f"{task}: invalid/empty timestamps or duplicates")
    if not np.isfinite(d[["actual", "predicted"]].to_numpy()).all():
        raise ValueError(f"{task}: nonfinite measurement or prediction")
    d = d.sort_values("datetime").reset_index(drop=True)
    predictions_by_task[task] = d
    print(task, len(d), d["datetime"].min(), "to", d["datetime"].max())

,Task,Selected on validation,MAE,RMSE,R2,Baseline_R2
0,NO2_next_hour,LightGBM,2.307140,3.566494,0.761870,0.627745
1,PM10_next_hour,LightGBM,4.178960,5.774977,0.332703,-0.023298
2,PM10_next_3h_average,LightGBM,3.053510,4.290585,0.440644,0.188021
3,PM10_next_24h_average,LightGBM,2.507151,3.356489,0.387174,0.172452


NO2_next_hour 16495 2019-09-24 09:00:00 to 2023-01-31 23:00:00
PM10_next_hour 16432 2019-09-24 09:00:00 to 2023-01-31 23:00:00
PM10_next_3h_average 15520 2019-09-24 09:00:00 to 2023-01-31 23:00:00
PM10_next_24h_average 10167 2019-09-24 09:00:00 to 2023-01-31 23:00:00


                Build CSVs matching your existing Streamlit queries


In [7]:
def label_predictions(task):
    d = predictions_by_task[task].copy()
    m = summary.loc[summary["Task"].eq(task), "Selected on validation"].iloc[0]
    d["model"] = str(m)
    return d

hourly = pd.concat([
    label_predictions("NO2_next_hour").assign(pollutant="NO2"),
    label_predictions("PM10_next_hour").assign(pollutant="PM10"),
], ignore_index=True)[["datetime", "pollutant", "model", "actual", "predicted"]]
averaged = pd.concat([
    label_predictions("PM10_next_3h_average").assign(horizon_hours=3),
    label_predictions("PM10_next_24h_average").assign(horizon_hours=24),
], ignore_index=True)[["datetime", "horizon_hours", "model", "actual", "predicted"]]

assert not hourly.duplicated(["datetime", "pollutant"]).any()
assert not averaged.duplicated(["datetime", "horizon_hours"]).any()
assert set(hourly["pollutant"]) == {"NO2", "PM10"}
assert set(averaged["horizon_hours"]) == {3, 24}

hourly_path = OUTPUT_DIR / "predictions_validation_revised.csv"
averaged_path = OUTPUT_DIR / "predictions_pm10_avg_validation_revised.csv"
hourly.to_csv(hourly_path, index=False)
averaged.to_csv(averaged_path, index=False)
print("Prepared ONLY local files; Supabase unchanged:")
print(hourly_path, "rows", len(hourly))
print(averaged_path, "rows", len(averaged))
display(hourly.groupby(["pollutant", "model"]).size().rename("rows").reset_index())
display(averaged.groupby(["horizon_hours", "model"]).size().rename("rows").reset_index())

Prepared ONLY local files; Supabase unchanged:
database_upload_review\predictions_validation_revised.csv rows 32927
database_upload_review\predictions_pm10_avg_validation_revised.csv rows 25687


,pollutant,model,rows
0,NO2,LightGBM,16495
1,PM10,LightGBM,16432


,horizon_hours,model,rows
0,3,LightGBM,15520
1,24,LightGBM,10167


         Verify revised prediction metrics match the selected report metrics


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
for task, d in predictions_by_task.items():
    row = summary.set_index("Task").loc[task]
    mae = mean_absolute_error(d["actual"], d["predicted"])
    rmse = np.sqrt(mean_squared_error(d["actual"], d["predicted"]))
    r2 = r2_score(d["actual"], d["predicted"])
    if not all(np.isclose([mae, rmse, r2], [row["MAE"], row["RMSE"], row["R2"]], atol=1e-6)):
        raise ValueError(f"{task}: prediction CSV does not match revised metrics. Stop and investigate.")
    print(f"{task}: matches revised results — MAE {mae:.3f}, RMSE {rmse:.3f}, R² {r2:.3f}")

NO2_next_hour: matches revised results — MAE 2.307, RMSE 3.566, R² 0.762
PM10_next_hour: matches revised results — MAE 4.179, RMSE 5.775, R² 0.333
PM10_next_3h_average: matches revised results — MAE 3.054, RMSE 4.291, R² 0.441
PM10_next_24h_average: matches revised results — MAE 2.507, RMSE 3.356, R² 0.387


             Connect to PostgreSQL and inspect existing tables (read only)


In [11]:
from sqlalchemy import create_engine, text, inspect
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("Set DATABASE_URL securely in .env before database verification")
engine = create_engine(DATABASE_URL, pool_pre_ping=True)
with engine.connect() as conn:
    for table in ("air_quality", "wind_direction", "predictions", "predictions_pm10_avg"):
        count = conn.execute(text(f'SELECT COUNT(*) FROM public."{table}"')).scalar_one()
        print(f"{table}: {count:,} existing rows")
    rls = conn.execute(text("""SELECT relname, relrowsecurity FROM pg_class
        JOIN pg_namespace ON pg_namespace.oid = pg_class.relnamespace
        WHERE nspname='public' AND relname IN
        ('air_quality','wind_direction','predictions','predictions_pm10_avg')
        ORDER BY relname""")).fetchall()
    print("RLS enabled:", rls)

inspector = inspect(engine)
for table, frame in (("predictions", hourly), ("predictions_pm10_avg", averaged)):
    existing_cols = {c["name"] for c in inspector.get_columns(table, schema="public")}
    missing = set(frame.columns) - existing_cols
    if missing:
        raise ValueError(f"{table}: existing DB table lacks columns {missing}; do not publish yet")
    print(table, "matching required columns confirmed")

air_quality: 150,432 existing rows
wind_direction: 4,325 existing rows
predictions: 50,347 existing rows
predictions_pm10_avg: 56,964 existing rows
RLS enabled: [('air_quality', True), ('predictions', True), ('predictions_pm10_avg', True), ('wind_direction', True)]
predictions matching required columns confirmed
predictions_pm10_avg matching required columns confirmed


        Back up existing prediction tables before any changes (read only)



In [13]:
backup_dir = OUTPUT_DIR / "private_database_backups"
backup_dir.mkdir(parents=True, exist_ok=True)
backup_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup_counts = {}
for table in ("predictions", "predictions_pm10_avg"):
    old = pd.read_sql_query(text(f'SELECT * FROM public."{table}"'), engine)
    target = backup_dir / f"{table}_before_update_{backup_tag}.csv"
    old.to_csv(target, index=False)
    backup_counts[table] = len(old)
    print("Backed up:", table, "rows:", len(old), "file:", target)
print("Backup complete. Verify files before enabling publishing.")

Backed up: predictions rows: 50347 file: database_upload_review\private_database_backups\predictions_before_update_20261030T115436Z.csv
Backed up: predictions_pm10_avg rows: 56964 file: database_upload_review\private_database_backups\predictions_pm10_avg_before_update_20261030T115436Z.csv
Backup complete. Verify files before enabling publishing.


        OPTIONAL: publish revised prediction tables



In [32]:
print("Hourly predictions:")
print(hourly.shape)
print(hourly.columns.tolist())
display(hourly.head())

print("Average predictions:")
print(averaged.shape)
print(averaged.columns.tolist())
display(averaged.head())

Hourly predictions:
(32927, 5)
['datetime', 'pollutant', 'model', 'actual', 'predicted']


,datetime,pollutant,model,actual,predicted
0,2019-09-24 09:00:00,NO2,LightGBM,2.8,4.350399
1,2019-09-24 10:00:00,NO2,LightGBM,1.3,2.182004
2,2019-09-24 11:00:00,NO2,LightGBM,1.0,1.319764
3,2019-09-24 12:00:00,NO2,LightGBM,1.3,1.229290
4,2019-09-24 13:00:00,NO2,LightGBM,0.8,1.522672


Average predictions:
(25687, 5)
['datetime', 'horizon_hours', 'model', 'actual', 'predicted']


,datetime,horizon_hours,model,actual,predicted
0,2019-09-24 09:00:00,3,LightGBM,4.800000,8.015524
1,2019-09-24 10:00:00,3,LightGBM,5.666667,7.668019
2,2019-09-24 11:00:00,3,LightGBM,10.200000,9.098921
3,2019-09-24 12:00:00,3,LightGBM,17.100000,8.535417
4,2019-09-24 13:00:00,3,LightGBM,15.533333,8.330075


In [34]:
PUBLISH_REVISED_PREDICTIONS = True # Manually set True ONLY after backups + approval.

if not PUBLISH_REVISED_PREDICTIONS:
    print("DRY RUN: No changes to Supabase. Revised CSVs are ready for review.")
else:
    confirm = input('Type PUBLISH REVISED PREDICTIONS to update two tables: ')
    if confirm != "PUBLISH REVISED PREDICTIONS":
        raise RuntimeError("Publish cancelled")
    with engine.begin() as conn:  # Transaction; rolls back on failure.
        for name, payload in (("predictions", hourly), ("predictions_pm10_avg", averaged)):
            # Preserve all existing table structures/security policies (never if_exists='replace').
            conn.execute(text(f'DELETE FROM public."{name}"'))
            payload.to_sql(name, conn, schema="public", if_exists="append",
                           index=False, chunksize=1000, method="multi")
            actual_count = conn.execute(text(f'SELECT COUNT(*) FROM public."{name}"')).scalar_one()
            if actual_count != len(payload):
                raise RuntimeError(f"{name}: row-count mismatch; transaction will roll back")
            rls = conn.execute(text("""SELECT relrowsecurity FROM pg_class c
                JOIN pg_namespace n ON n.oid=c.relnamespace
                WHERE n.nspname='public' AND c.relname=:name"""), {"name": name}).scalar_one()
            if not rls:
                raise RuntimeError(f"{name}: RLS is disabled; transaction will roll back")
            print(f"Published {actual_count:,} rows to {name}; RLS still enabled")
    print("Publication completed. Run the API read test below and verify Streamlit.")

Type PUBLISH REVISED PREDICTIONS to update two tables:  PUBLISH REVISED PREDICTIONS


Published 32,927 rows to predictions; RLS still enabled
Published 25,687 rows to predictions_pm10_avg; RLS still enabled
Publication completed. Run the API read test below and verify Streamlit.


    REST API read test using an appropriate read-only client key

In [36]:
from supabase import create_client
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise RuntimeError("Set SUPABASE_URL and a non-privileged SUPABASE_KEY in .env")
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])
res = (sb.table("air_quality").select("datetime,pm10,no2")
       .gte("datetime", "2019-07-01").lt("datetime", "2019-07-02")
       .order("datetime").limit(100).execute())
print("Historical records returned for 2019-07-01:", len(res.data))
for table in ("predictions", "predictions_pm10_avg"):
    reply = sb.table(table).select("*").limit(3).execute()
    print(f"{table}: API returned {len(reply.data)} sample rows")

Historical records returned for 2019-07-01: 24
predictions: API returned 3 sample rows
predictions_pm10_avg: API returned 3 sample rows


             Non-destructive access-control checks

In [30]:
with engine.connect() as conn:
    policies = pd.read_sql_query(text("""SELECT tablename, policyname, roles, cmd
        FROM pg_policies WHERE schemaname='public'
        AND tablename IN ('air_quality','predictions','predictions_pm10_avg','wind_direction')
        ORDER BY tablename, policyname"""), conn)
    display(policies)
    grants = pd.read_sql_query(text("""SELECT table_name, grantee, privilege_type
        FROM information_schema.role_table_grants
        WHERE table_schema='public'
        AND table_name IN ('air_quality','predictions','predictions_pm10_avg','wind_direction')
        AND grantee IN ('anon','authenticated')
        ORDER BY table_name, grantee, privilege_type"""), conn)
    display(grants)
print("Review for unexpected INSERT/UPDATE/DELETE grants and permissive policies.")

,tablename,policyname,roles,cmd
0,air_quality,read only,[anon],SELECT
1,predictions,read only,[anon],SELECT
2,predictions_pm10_avg,read only,[anon],SELECT
3,wind_direction,read only,[anon],SELECT


,table_name,grantee,privilege_type
0,air_quality,anon,SELECT
1,air_quality,authenticated,SELECT
2,predictions,anon,SELECT
3,predictions,authenticated,SELECT
4,predictions_pm10_avg,anon,SELECT
5,predictions_pm10_avg,authenticated,SELECT
6,wind_direction,anon,SELECT
7,wind_direction,authenticated,SELECT


Review for unexpected INSERT/UPDATE/DELETE grants and permissive policies.
